# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kavanaaykavna/1stmlassignment/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

My baseline rule prioritizes pages that are both stale and have a low click-through rate (CTR). The assumption is that older pages with poor CTR are good candidates for content refresh or title/meta optimization.

The rule is intended as a simple decision-support baseline. It is not a predictive model and does not use any future information.

### Reason Codes

| Reason Code | Meaning | Action |
|-------------|---------|--------|
| STALE_LOW_CTR | Page is older than 180 days and CTR is below 3% | Refresh Content |
| STALE | Page is older than 180 days | Review Content |
| LOW_CTR | CTR is below 3% | Improve Title / Meta |
| OK | No major issues detected | No Action |

In [4]:
import pandas as pd
import os

# Load dataset
df = pd.read_csv("content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:\n")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
import pandas as pd
import os

# Load dataset
df = pd.read_csv("content_refresh_anonymized.csv")

# Baseline score
df["baseline_score"] = 0

# Rule 1: Stale content
df.loc[df["days_since_last_update"] > 180, "baseline_score"] += 1

# Rule 2: Low CTR
df.loc[df["ctr"] < 0.03, "baseline_score"] += 1

# Reason Codes
def get_reason(row):
    stale = row["days_since_last_update"] > 180
    low_ctr = row["ctr"] < 0.03

    if stale and low_ctr:
        return "STALE_LOW_CTR"
    elif stale:
        return "STALE"
    elif low_ctr:
        return "LOW_CTR"
    else:
        return "OK"

df["reason_code"] = df.apply(get_reason, axis=1)

# Action Labels
action_map = {
    "STALE_LOW_CTR": "Refresh Content",
    "STALE": "Review Content",
    "LOW_CTR": "Improve Title/Meta",
    "OK": "No Action"
}

df["action"] = df["reason_code"].map(action_map)

# Ranked Queue
queue = df.sort_values(
    by="baseline_score",
    ascending=False
)[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]]

# Show Top 20
print(queue.head(20))

# Save CSV
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("✅ baseline_action_score.csv created successfully!")


                 content_id  baseline_score    reason_code           action
22980  content_f328d0e4e22b               2  STALE_LOW_CTR  Refresh Content
26981  content_15fe075b97bc               2  STALE_LOW_CTR  Refresh Content
25578  content_e2181e471a9c               2  STALE_LOW_CTR  Refresh Content
15608  content_06e19c6486b0               2  STALE_LOW_CTR  Refresh Content
6119   content_cd27391ecd03               2  STALE_LOW_CTR  Refresh Content
27378  content_958a46db26bd               2  STALE_LOW_CTR  Refresh Content
8125   content_ccf25ed65a99               2  STALE_LOW_CTR  Refresh Content
1361   content_74961b456728               2  STALE_LOW_CTR  Refresh Content
2653   content_1d10143d4e52               2  STALE_LOW_CTR  Refresh Content
744    content_a98703986e70               2  STALE_LOW_CTR  Refresh Content
29830  content_1a6977ff1ef1               2  STALE_LOW_CTR  Refresh Content
9476   content_30eb41dff556               2  STALE_LOW_CTR  Refresh Content
6421   conte

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
top20 = queue.head(20)

for i, row in enumerate(top20.itertuples(index=False), start=1):
    print(f"""
Rank {i}

Action:
{row.action}

Reason Code:
{row.reason_code}

Confidence:
Medium

What would make it wrong?
This page may be seasonal, have low search demand, or require additional business context before taking action.

--------------------------------------------------------
""")


Rank 1

Action:
Refresh Content

Reason Code:
STALE_LOW_CTR

Confidence:
Medium

What would make it wrong?
This page may be seasonal, have low search demand, or require additional business context before taking action.

--------------------------------------------------------


Rank 2

Action:
Refresh Content

Reason Code:
STALE_LOW_CTR

Confidence:
Medium

What would make it wrong?
This page may be seasonal, have low search demand, or require additional business context before taking action.

--------------------------------------------------------


Rank 3

Action:
Refresh Content

Reason Code:
STALE_LOW_CTR

Confidence:
Medium

What would make it wrong?
This page may be seasonal, have low search demand, or require additional business context before taking action.

--------------------------------------------------------


Rank 4

Action:
Refresh Content

Reason Code:
STALE_LOW_CTR

Confidence:
Medium

What would make it wrong?
This page may be seasonal, have low search demand, or r

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



## Weak Picks

The baseline rule is intentionally simple, so some recommendations may be incorrect.

Possible weak picks include:

- **Seasonal content:** Some pages naturally receive lower traffic during certain times of the year, so refreshing them may not improve performance.
- **Low search demand:** Pages targeting low-volume keywords may have low CTR even when the content is relevant.
- **Recently updated pages:** A page may have been updated recently but not yet shown improved performance in the available data.
- **Pages with few impressions:** CTR can be unstable when the number of impressions is very small.
- **Business-specific pages:** Some pages are intentionally niche and may not require optimization despite a low score.

## Leakage Check

No product flags, future-window information, or target labels were used when building the baseline rule.

The baseline score only uses the following observable signals:

- `days_since_last_update`
- `ctr`

The following fields were **not** used because they could introduce target leakage or future information:

- `trend_direction`
- `trend_pct`
- Any product-generated flags or labels

Therefore, the baseline is intended as a simple **decision-support rule** without information leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.